In [ ]:
cd ..

In [97]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.churn_label_generator import generate_churn_labels



In [98]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")


In [ ]:

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')


In [ ]:
silver_money_events = silver_money_events.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('event_ts')))
sessions_silver = sessions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('session_date')))
transactions_silver = transactions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date),F.to_date(F.col('transaction_ts'))))

In [ ]:
df_churn = generate_churn_labels(sessions_silver, config_)
df_churn = (
    df_churn
    .join(
        players_silver.select("player_idx"),
        on="player_idx",
        how="inner"
    )
)
#df_churn.write.mode("append").parquet("./data/bronze/churn_labels")


In [ ]:
player_snapshot = (players_silver
                   .select('player_idx','country','age_bucket','device_type','acquisition_channel', 'registration_date', 'risk_segment', 'lifecycle_stage', F.col('current_balance').alias('current_balance'))
                   .join(df_churn.select('player_idx','first_session_date','last_session_date'),
                         on='player_idx',
                         how='left')
                         .withColumn('days_since_last_session', 
                                     F.date_diff(F.lit(config_.end_date), F.col('last_session_date')))
                         .drop('reference_date')

)
player_snapshot.show()

## Window functions for rolling computations 7days or 30days ago

In [ ]:
'''
sessions_silver_sec = (sessions_silver
                   .withColumn('session_date_days', 
                F.datediff(F.col("session_date"), F.lit("1970-01-01"))
)
)

window_7d = (Window.partitionBy('player_id').orderBy('session_date_days').rangeBetween(-7,0))
window_30d = (Window.partitionBy('player_id').orderBy('session_date_days').rangeBetween(-30,0))
sessions_silver_sec = (sessions_silver_sec
            .withColumn('num_sessions_7d', F.count('session_id').over(window_7d))
            .withColumn('num_sessions_30d', F.count('session_id').over(window_30d))
            .withColumn('avg_sessions_duration_30d', F.avg('session_duration_sec').over(window_30d))
)

((sessions_silver_sec.groupBy('player_id')
 .agg(
     F.max('session_date_days').alias('session_date_days'))
)
.join(sessions_silver_sec.select('player_id', 'session_date_days','session_date','num_sessions_7d','num_sessions_30d','avg_sessions_duration_30d'), on=['player_id', 'session_date_days'], how='inner')
.show()
)

'''

# player_behavior

### player_behavior based on sessions activity

In [ ]:
player_behavior = (sessions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('session_date')))
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_idx')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, 1).otherwise(0)).alias('sessions_7d'),
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_game_result_7d'),

            F.count('*').alias('sesions_30d'),
            F.avg(F.col('session_duration_sec')).cast('int').alias('avg_session_duration_sec_30d'),
            F.avg(F.col('total_bet_amount')).alias('avg_bet_amount_30d'),
            F.sum(F.col('signed_amount')).alias('net_game_result_30d'),
 )
                
)
player_behavior.show()

In [96]:
player_snapshot.show()

+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+-----------------+-----------------------+
|player_idx|country|age_bucket|device_type|acquisition_channel|registration_date|risk_segment|lifecycle_stage|   current_balance|last_session_date|days_since_last_session|
+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+-----------------+-----------------------+
|         0|     EN|     25-34|    desktop|               paid|       2024-03-15|         low|        at_risk|            148.56|       2024-06-17|                     13|
|      1002|     GR|     25-34|    desktop|          affiliate|       2024-01-09|      medium|        engaged|             76.07|       2024-06-29|                      1|
|     10000|     EN|     25-34|     mobile|            organic|       2024-04-14|         low|        engaged|224.10999999999999|       2024

### player behavior based on any money event (financial + bet)

In [ ]:

### Correct ###
silver_money_events_net = (silver_money_events
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_idx')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_amount_result_7d'),
            F.sum(F.col('signed_amount')).alias('net_amount_result_30d'),
 )        
)

silver_money_events_net.show()

In [ ]:
w = Window.partitionBy("player_idx").orderBy(F.col("event_ts").desc())
player_30d = (silver_money_events
 .filter(F.col('days_diff') >=30)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_30d_act= (players_silver
                   .select('player_idx','current_balance')
                   .join(player_30d.select('player_idx','balance_after_txn'),
                         on='player_idx',
                         how='left')
                             .withColumn(
                              "balance_30d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop( 'current_balance', 'balance_after_txn')
)
player_30d_act.show()

In [ ]:
w = Window.partitionBy("player_idx").orderBy(F.col("event_ts").desc())
player_7d = (silver_money_events
 .filter(F.col('days_diff') >=7)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_7d_act= (players_silver
                   .select('player_idx','current_balance')
                   .join(player_7d.select('player_idx',F.col('balance_after_txn')),
                         on='player_idx',
                         how='left')
                             .withColumn(
                              "balance_7d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop('current_balance', 'balance_after_txn')
)
player_7d_act.show()

In [ ]:
transaction_behavior = (transactions_silver
.filter(F.col('days_diff')<=30)
.groupBy(F.col('player_idx'))
.agg(
    F.sum(F.when((F.col('transaction_type')=='withdrawal') & (F.col('success_flag')==False),1).otherwise(0)).alias('failed_withdrawals_30d'),
    F.sum(F.when(F.col('transaction_type')=='deposit',1).otherwise(0)).alias('deposit_count_30d'),
    F.sum(F.when(F.col('transaction_type')=='withdrawal',1).otherwise(0)).alias('withdrawal_count_30d'),
)
.withColumn(        'withdrawal_ratio',
    F.when(
            F.col("deposit_count_30d") > 0,
            F.col("withdrawal_count_30d") / F.col("deposit_count_30d")
        ).otherwise(F.lit(0.0))
)
)
transaction_behavior.show()

In [ ]:
gold_player_behavior = (players_silver.select('player_idx')
                        .join(silver_money_events_net, how='left', on='player_idx') 
                        .join(transaction_behavior, how='left', on='player_idx') 
                        .join(player_behavior, how='left', on='player_idx') 
                        .join(player_7d_act, how='left', on='player_idx') 
                        .join(player_30d_act, how='left', on='player_idx') 

)
gold_player_behavior.show()

In [ ]:
df_churn.printSchema()



In [ ]:
sessions_silver.show(2)

In [ ]:

date_range = spark.sql(
    f"SELECT explode(sequence(to_date('{config_.start_date}'), to_date('{config_.end_date}'), interval 1 day)) AS event_date"
)

df = players_silver.crossJoin(date_range)
df = (
    df
    .withColumn(
    "daily_sessions",
    F.when(
        F.col("lifecycle_stage") == "engaged",
        F.floor(-F.log(1 - F.rand(seed=42)) * F.lit(config_.active_lambda))
    ).when(
        F.col("lifecycle_stage") == "at_risk",
        F.floor(-F.log(1 - F.rand(seed=42)) * F.lit(config_.at_risk_lambda))
    ).otherwise(F.lit(0))
)
    .filter(F.col("daily_sessions") > 0)
    .withColumn("session_seq", F.explode(F.sequence(F.lit(1), F.col("daily_sessions"))))
    .withColumn("session_id", F.expr("uuid()"))
    .withColumn("game_id", F.concat(F.lit("G"), (F.rand() * 200).cast("int")))
    .withColumn("session_duration_sec", (F.rand() * 3600).cast("int"))
    .withColumn("bet_count", (F.rand() * 20 + 1).cast("int"))
    .withColumn("total_bet_amount", F.round(F.rand() * 100 + 1, 2))
    .withColumn("total_win_amount", F.round(F.rand() * 120 + 1 , 2))
    .select(
        "session_id",
        "player_idx",
        'daily_sessions',
        "game_id",
        F.col("event_date").alias("session_date"),
        "session_duration_sec",
        "bet_count",
        "total_bet_amount",
        "total_win_amount",
        "device_type"
    )
)
            


In [ ]:
df.filter(F.col('bet_count')==0).show()

In [76]:
date_range = spark.sql(
    f"SELECT explode(sequence(to_date('{config_.start_date}'), to_date('{config_.end_date}'), interval 1 day)) AS event_date"
)

df_pl_date = players_silver.select('player_idx').crossJoin(date_range)

df_num_of_sessions = (sessions_silver
.groupBy('player_idx', 'session_date')
.agg(F.count('*').alias('num_of_session')))
#.orderBy('player_idx', 'session_date' ))

df_sessions = (df_pl_date
.join(df_num_of_sessions
      .select('num_of_session', F.col('session_date').alias('event_date'), 'player_idx'),
        how='left', on=['player_idx','event_date'])
)


In [90]:
df_sessions_sec = (df_sessions
                   .withColumn('session_date_days', 
                F.datediff(F.col("event_date"), F.lit("1970-01-01"))
)
)

window_7d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(-7,0))
window_30d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(-30,0))

sessions_silver_sec = (df_sessions_sec
            .withColumn('num_sessions_7d', 
                        F.sum(F.when(F.col("num_of_session") > 0, 1).otherwise(0)).over(window_7d))
            .withColumn('churn_7d', F.when(F.col('num_sessions_7d')==0, True).otherwise(False))
            .withColumn('num_sessions_30d', 
                        F.sum(F.when(F.col('num_of_session') > 0, 1).otherwise(0)).over(window_30d))
           .withColumn('churn_30d', F.when(F.col('num_sessions_30d')==0, True).otherwise(False))
)

In [99]:
df_sessions_sec.show(5)

+----------+----------+--------------+-----------------+
|player_idx|event_date|num_of_session|session_date_days|
+----------+----------+--------------+-----------------+
|         1|2024-01-05|          NULL|            19727|
|     10007|2024-01-03|          NULL|            19725|
|         1|2024-01-06|          NULL|            19728|
|     10002|2024-01-02|          NULL|            19724|
|     10007|2024-01-06|          NULL|            19728|
+----------+----------+--------------+-----------------+
only showing top 5 rows


In [93]:
sessions_silver_sec.filter(F.col('player_idx')==87).orderBy('event_date').show()

+----------+----------+--------------+-----------------+---------------+--------+----------------+---------+
|player_idx|event_date|num_of_session|session_date_days|num_sessions_7d|churn_7d|num_sessions_30d|churn_30d|
+----------+----------+--------------+-----------------+---------------+--------+----------------+---------+
|        87|2024-01-01|          NULL|            19723|              0|    true|               0|     true|
|        87|2024-01-02|          NULL|            19724|              0|    true|               0|     true|
|        87|2024-01-03|          NULL|            19725|              0|    true|               0|     true|
|        87|2024-01-04|          NULL|            19726|              0|    true|               0|     true|
|        87|2024-01-05|          NULL|            19727|              0|    true|               0|     true|
|        87|2024-01-06|          NULL|            19728|              0|    true|               0|     true|
|        87|2024-01

In [94]:
sessions_silver.filter(F.col('player_idx')==87).orderBy('session_date').show()

+--------------------+-------+--------------------+---------+----------------+----------------+-----------+------------+-------------+------------------+----------+---------+
|          session_id|game_id|session_duration_sec|bet_count|total_bet_amount|total_win_amount|device_type|session_date|signed_amount| balance_after_txn|player_idx|days_diff|
+--------------------+-------+--------------------+---------+----------------+----------------+-----------+------------+-------------+------------------+----------+---------+
|1b1a5467-a6c4-473...|   G157|                2654|       10|           19.78|           59.55|    desktop|  2024-02-05|        59.55|            117.77|        87|      146|
|f948df81-0208-4c3...|     G5|                1484|       17|           81.91|           25.91|    desktop|  2024-02-06|        25.91|            143.68|        87|      145|
|495f5322-bdba-456...|   G103|                1530|        2|           54.43|           32.09|    desktop|  2024-02-07|     

In [92]:
sessions_silver_sec.filter(F.col('churn_7d')==True).filter(F.col('event_date')>'2024-01-07') .orderBy('event_date').show()

+----------+----------+--------------+-----------------+---------------+--------+----------------+---------+
|player_idx|event_date|num_of_session|session_date_days|num_sessions_7d|churn_7d|num_sessions_30d|churn_30d|
+----------+----------+--------------+-----------------+---------------+--------+----------------+---------+
|        87|2024-01-08|          NULL|            19730|              0|    true|               0|     true|
|        91|2024-01-08|          NULL|            19730|              0|    true|               0|     true|
|      1092|2024-01-08|          NULL|            19730|              0|    true|               0|     true|
|       267|2024-01-08|          NULL|            19730|              0|    true|               0|     true|
|        83|2024-01-08|          NULL|            19730|              0|    true|               0|     true|
|        93|2024-01-08|          NULL|            19730|              0|    true|               0|     true|
|      2203|2024-01

In [ ]:
df_sessions.orderBy('player_idx',).filter(F.col('event_date')=='2024-04-26').show(30, truncate=False)

In [ ]:
df_num_of_sessions.orderBy('player_idx', 'session_date').show()